## <center> Predicting Movie Rental Durations <center>

In [1]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import datetime as dt
import operator as op

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

In [2]:
# Import the rental_info.csv dataset
rental_df = pd.read_csv("datasets/rental_info.csv", parse_dates=["rental_date", "return_date"])
rental_df.head()

,rental_date,return_date,amount,release_year,rental_rate,length,replacement_cost,special_features,NC-17,PG,PG-13,R,amount_2,length_2,rental_rate_2
0,2005-05-25 02:54:33+00:00,2005-05-28 23:40:33+00:00,2.99,2005.0,2.99,126.0,16.99,"{Trailers,""Behind the Scenes""}",0,0,0,1,8.9401,15876.0,8.9401
1,2005-06-15 23:19:16+00:00,2005-06-18 19:24:16+00:00,2.99,2005.0,2.99,126.0,16.99,"{Trailers,""Behind the Scenes""}",0,0,0,1,8.9401,15876.0,8.9401
2,2005-07-10 04:27:45+00:00,2005-07-17 10:11:45+00:00,2.99,2005.0,2.99,126.0,16.99,"{Trailers,""Behind the Scenes""}",0,0,0,1,8.9401,15876.0,8.9401
3,2005-07-31 12:06:41+00:00,2005-08-02 14:30:41+00:00,2.99,2005.0,2.99,126.0,16.99,"{Trailers,""Behind the Scenes""}",0,0,0,1,8.9401,15876.0,8.9401
4,2005-08-19 12:30:04+00:00,2005-08-23 13:35:04+00:00,2.99,2005.0,2.99,126.0,16.99,"{Trailers,""Behind the Scenes""}",0,0,0,1,8.9401,15876.0,8.9401


In [3]:
# Review the data frame info
rental_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15861 entries, 0 to 15860
Data columns (total 15 columns):
 #   Column            Non-Null Count  Dtype              
---  ------            --------------  -----              
 0   rental_date       15861 non-null  datetime64[ns, UTC]
 1   return_date       15861 non-null  datetime64[ns, UTC]
 2   amount            15861 non-null  float64            
 3   release_year      15861 non-null  float64            
 4   rental_rate       15861 non-null  float64            
 5   length            15861 non-null  float64            
 6   replacement_cost  15861 non-null  float64            
 7   special_features  15861 non-null  object             
 8   NC-17             15861 non-null  int64              
 9   PG                15861 non-null  int64              
 10  PG-13             15861 non-null  int64              
 11  R                 15861 non-null  int64              
 12  amount_2          15861 non-null  float64            
 13  l

In [4]:
# Shape of the data frame
rental_df.shape

(15861, 15)

In [5]:
# Create a 'rental_length_days' columns by subtracting the rental_date from the return_date
rental_df["rental_length_days"] = rental_df["return_date"] - rental_df["rental_date"]
rental_df["rental_length_days"] = rental_df["rental_length_days"].dt.days

In [6]:
# Examine the first few lines of the rental_df data frame
rental_df.head()

,rental_date,return_date,amount,release_year,rental_rate,length,replacement_cost,special_features,NC-17,PG,PG-13,R,amount_2,length_2,rental_rate_2,rental_length_days
0,2005-05-25 02:54:33+00:00,2005-05-28 23:40:33+00:00,2.99,2005.0,2.99,126.0,16.99,"{Trailers,""Behind the Scenes""}",0,0,0,1,8.9401,15876.0,8.9401,3
1,2005-06-15 23:19:16+00:00,2005-06-18 19:24:16+00:00,2.99,2005.0,2.99,126.0,16.99,"{Trailers,""Behind the Scenes""}",0,0,0,1,8.9401,15876.0,8.9401,2
2,2005-07-10 04:27:45+00:00,2005-07-17 10:11:45+00:00,2.99,2005.0,2.99,126.0,16.99,"{Trailers,""Behind the Scenes""}",0,0,0,1,8.9401,15876.0,8.9401,7
3,2005-07-31 12:06:41+00:00,2005-08-02 14:30:41+00:00,2.99,2005.0,2.99,126.0,16.99,"{Trailers,""Behind the Scenes""}",0,0,0,1,8.9401,15876.0,8.9401,2
4,2005-08-19 12:30:04+00:00,2005-08-23 13:35:04+00:00,2.99,2005.0,2.99,126.0,16.99,"{Trailers,""Behind the Scenes""}",0,0,0,1,8.9401,15876.0,8.9401,4


In [7]:
# Examine the values of the 'special_features' column
rental_df["special_features"].unique()

array(['{Trailers,"Behind the Scenes"}', '{Trailers}',
       '{Commentaries,"Behind the Scenes"}', '{Trailers,Commentaries}',
       '{"Deleted Scenes","Behind the Scenes"}',
       '{Commentaries,"Deleted Scenes","Behind the Scenes"}',
       '{Trailers,Commentaries,"Deleted Scenes"}',
       '{"Behind the Scenes"}',
       '{Trailers,"Deleted Scenes","Behind the Scenes"}',
       '{Commentaries,"Deleted Scenes"}', '{Commentaries}',
       '{Trailers,Commentaries,"Behind the Scenes"}',
       '{Trailers,"Deleted Scenes"}', '{"Deleted Scenes"}',
       '{Trailers,Commentaries,"Deleted Scenes","Behind the Scenes"}'],
      dtype=object)

In [8]:
# Delete the '}', '{', and '"' in 'special_features'
rental_df["special_features"] = rental_df["special_features"].str.replace('{', '')
rental_df["special_features"] = rental_df["special_features"].str.replace('}', '')
rental_df["special_features"] = rental_df["special_features"].str.replace('"', '')

In [9]:
# Convert values in the 'special_features' column to lists
rental_df["special_features"] = rental_df["special_features"].to_list()

In [10]:
# Create the column 'deleted_scenes' and assign '1' to rows that contain the phrase 'Deleted Scenes'
for index, value in rental_df["special_features"].items():
    if op.contains(value, 'Deleted'):
        rental_df.loc[index, "deleted_scenes"] = 1
    else:
        rental_df.loc[index, "deleted_scenes"] = 0

In [11]:
# Sample the data frame to ensure the 'deleted_scenes' column contains correct values
rental_df.sample(n=15, replace = False)

,rental_date,return_date,amount,release_year,rental_rate,length,replacement_cost,special_features,NC-17,PG,PG-13,R,amount_2,length_2,rental_rate_2,rental_length_days,deleted_scenes
12452,2005-07-27 10:14:36+00:00,2005-08-02 09:18:36+00:00,3.99,2009.0,2.99,168.0,11.99,"Trailers,Commentaries,Deleted Scenes",1,0,0,0,15.9201,28224.0,8.9401,5,1.0
14705,2005-07-29 19:26:31+00:00,2005-08-06 23:05:31+00:00,6.99,2006.0,2.99,151.0,15.99,"Trailers,Behind the Scenes",0,1,0,0,48.8601,22801.0,8.9401,8,0.0
15495,2005-08-02 13:25:31+00:00,2005-08-08 12:10:31+00:00,2.99,2006.0,0.99,171.0,28.99,"Trailers,Commentaries,Deleted Scenes",0,0,1,0,8.9401,29241.0,0.9801,5,1.0
633,2005-08-18 19:12:17+00:00,2005-08-19 16:26:17+00:00,4.99,2010.0,4.99,77.0,23.99,Trailers,1,0,0,0,24.9001,5929.0,24.9001,0,0.0
2750,2005-07-09 02:49:37+00:00,2005-07-18 01:03:37+00:00,5.99,2009.0,0.99,103.0,27.99,"Trailers,Commentaries,Deleted Scenes",1,0,0,0,35.8801,10609.0,0.9801,8,1.0
1716,2005-08-22 12:42:45+00:00,2005-08-24 09:20:45+00:00,4.99,2007.0,4.99,91.0,16.99,"Deleted Scenes,Behind the Scenes",0,0,1,0,24.9001,8281.0,24.9001,1,1.0
6881,2005-06-21 05:19:37+00:00,2005-06-29 10:45:37+00:00,4.99,2004.0,2.99,124.0,16.99,Behind the Scenes,0,0,0,1,24.9001,15376.0,8.9401,8,0.0
5525,2005-08-20 19:39:00+00:00,2005-08-24 20:17:00+00:00,4.99,2009.0,4.99,176.0,19.99,"Deleted Scenes,Behind the Scenes",0,1,0,0,24.9001,30976.0,24.9001,4,1.0
3696,2005-06-19 10:55:01+00:00,2005-06-25 07:21:01+00:00,5.99,2004.0,4.99,73.0,11.99,Commentaries,0,0,0,1,35.8801,5329.0,24.9001,5,0.0
7596,2005-07-06 15:54:18+00:00,2005-07-09 14:48:18+00:00,0.99,2009.0,0.99,164.0,22.99,Deleted Scenes,0,0,0,1,0.9801,26896.0,0.9801,2,1.0


In [12]:
# Create the column 'behind_the_scenes' and assign '1' to rows that contain the phrase 'Behind the Scenes'
for index, value in rental_df["special_features"].items():
    if op.contains(value, "Behind"):
        rental_df.loc[index, "behind_the_scenes"] = 1
    else:
        rental_df.loc[index, "behind_the_scenes"] = 0

In [13]:
# Sample the data frame to ensure the 'behind_the_scenes' column contains correct values
rental_df.sample(n = 15, replace = False)

,rental_date,return_date,amount,release_year,rental_rate,length,replacement_cost,special_features,NC-17,PG,PG-13,R,amount_2,length_2,rental_rate_2,rental_length_days,deleted_scenes,behind_the_scenes
5398,2005-07-09 01:32:17+00:00,2005-07-15 00:55:17+00:00,6.99,2010.0,4.99,175.0,11.99,"Trailers,Commentaries,Behind the Scenes",1,0,0,0,48.8601,30625.0,24.9001,5,0.0,1.0
10010,2005-08-18 05:34:13+00:00,2005-08-24 08:38:13+00:00,5.99,2004.0,2.99,179.0,10.99,Behind the Scenes,0,0,0,0,35.8801,32041.0,8.9401,6,0.0,1.0
1705,2005-08-01 11:08:46+00:00,2005-08-08 07:47:46+00:00,7.99,2008.0,4.99,93.0,19.99,"Trailers,Commentaries,Deleted Scenes,Behind th...",0,0,0,0,63.8401,8649.0,24.9001,6,1.0,1.0
6382,2005-08-18 17:43:07+00:00,2005-08-21 11:53:07+00:00,0.99,2005.0,0.99,143.0,16.99,"Trailers,Commentaries,Behind the Scenes",0,0,1,0,0.9801,20449.0,0.9801,2,0.0,1.0
5860,2005-08-23 18:43:46+00:00,2005-08-27 20:43:46+00:00,5.99,2004.0,4.99,150.0,28.99,"Commentaries,Deleted Scenes",0,0,0,0,35.8801,22500.0,24.9001,4,1.0,0.0
3917,2005-05-31 01:11:19+00:00,2005-06-06 06:16:19+00:00,4.99,2005.0,0.99,179.0,10.99,"Trailers,Commentaries,Deleted Scenes",0,0,1,0,24.9001,32041.0,0.9801,6,1.0,0.0
12882,2005-07-28 05:47:20+00:00,2005-08-05 10:00:20+00:00,1.99,2006.0,0.99,165.0,18.99,"Commentaries,Deleted Scenes",0,0,0,0,3.9601,27225.0,0.9801,8,1.0,0.0
6612,2005-07-31 14:30:25+00:00,2005-08-03 18:58:25+00:00,2.99,2008.0,2.99,70.0,22.99,"Commentaries,Deleted Scenes,Behind the Scenes",0,0,0,1,8.9401,4900.0,8.9401,3,1.0,1.0
4809,2005-07-09 13:18:43+00:00,2005-07-14 16:59:43+00:00,1.99,2007.0,0.99,178.0,14.99,Trailers,0,0,1,0,3.9601,31684.0,0.9801,5,0.0,0.0
3739,2005-08-19 09:38:25+00:00,2005-08-24 03:52:25+00:00,2.99,2010.0,2.99,153.0,13.99,"Trailers,Commentaries,Deleted Scenes",0,0,1,0,8.9401,23409.0,8.9401,4,1.0,0.0
